Referenced chatgpt report: https://chatgpt.com/c/699d7631-62dc-8394-8feb-7760acfaaaab

# Gemini model with structure output

In [5]:
from dotenv import load_dotenv
load_dotenv(override=True)

import json
from pydantic import BaseModel, Field

from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

# --- 1) Define the structured output schema ---
# NOTE: Gemini API does not support additionalProperties. Avoid dict[str, str] or
# any free-form object; use fixed-shape models or list[str] instead.

class ChecklistOutput(BaseModel):
    """Structured checklist response. Use only fixed fields so the schema stays Gemini-compatible."""

    title: str = Field(description="Short title for the checklist.")
    summary: str = Field(description="One sentence on what the checklist achieves.")
    steps: list[str] = Field(description="Ordered steps to complete the task. Be concrete and actionable.")
    risks_or_notes: list[str] = Field(
        default_factory=list,
        description="Optional pitfalls, gotchas, or important notes.",
    )

# --- 2) Configure the agent ---
structured_agent = Agent(
    name="structured_output_agent",
    model="gemini-3.1-pro-preview",
    instruction=(
        "You are a strict JSON generator. Your reply must be valid JSON that matches the requested schema exactly.\n"
        "Do not wrap the JSON in markdown code fences or add any text before or after it.\n"
        "For checklists: give clear, numbered-style steps; avoid vague or duplicate items."
    ),
    output_schema=ChecklistOutput,
)

runner = InMemoryRunner(agent=structured_agent)

# --- 3) Run a sample prompt and parse the JSON result ---
async def demo():
    prompt = "Create a short checklist for setting up a Python project with uv and dotenv."
    events = await runner.run_debug(prompt)

    final_text = None
    for ev in events:
        if ev.is_final_response() and ev.content and ev.content.parts:
            final_text = "".join([p.text for p in ev.content.parts if p.text])
            break

    if not final_text:
        raise RuntimeError("No final text response found.")

    data = json.loads(final_text)
    obj = ChecklistOutput.model_validate(data)
    return obj

result = await demo()

# Print the whole result
print("--- Full result ---")
print(result)
print()

# Print one by one by key
print("--- By key ---")
print("title:", result.title)
print("summary:", result.summary)
print("steps:")
for i, step in enumerate(result.steps, 1):
    print(f"  {i}. {step}")
print("risks_or_notes:", result.risks_or_notes)

result



 ### Created new session: debug_session_id

User > Create a short checklist for setting up a Python project with uv and dotenv.
structured_output_agent > {"title":"Python Project Setup with uv and dotenv","summary":"A streamlined guide to initializing a Python project, managing dependencies with uv, and configuring environment variables using python-dotenv.","steps":["Install uv globally if not already installed.","Create and navigate to your new project directory.","Run 'uv init' to generate the basic project structure including pyproject.toml.","Run 'uv venv' to create a virtual environment, then activate it.","Run 'uv add python-dotenv' to install the dotenv package and update your dependencies.","Create a '.env' file in the root of your project and add your configuration variables.","Add '.env' to your '.gitignore' file immediately to prevent accidental commits of sensitive data.","In your main Python file, import load_dotenv from dotenv and call it at the very top of your script.

ChecklistOutput(title='Python Project Setup with uv and dotenv', summary='A streamlined guide to initializing a Python project, managing dependencies with uv, and configuring environment variables using python-dotenv.', steps=['Install uv globally if not already installed.', 'Create and navigate to your new project directory.', "Run 'uv init' to generate the basic project structure including pyproject.toml.", "Run 'uv venv' to create a virtual environment, then activate it.", "Run 'uv add python-dotenv' to install the dotenv package and update your dependencies.", "Create a '.env' file in the root of your project and add your configuration variables.", "Add '.env' to your '.gitignore' file immediately to prevent accidental commits of sensitive data.", 'In your main Python file, import load_dotenv from dotenv and call it at the very top of your script.'], risks_or_notes=['Failing to add .env to .gitignore is a major security risk that can expose API keys.', 'Remember to activate the uv 

# Generate images with Nano Banana

Key parameters and options (from the official image docs)
Output modalities can be configured (image-only vs text+image). 
Aspect ratio options include: 1:1, 2:3, 3:2, 3:4, 4:3, 4:5, 5:4, 9:16, 16:9, 21:9. 
For gemini-3-pro-image-preview, you can set image_size to 1K, 2K, or 4K (uppercase K required). 
The docs also explicitly note the model may not always follow the exact number of images requested. 

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from google.genai import types
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

from IPython.display import display
from datetime import datetime

# Choose Nano Banana variants:
# - "gemini-2.5-flash-image"        (Nano Banana: fast)
# - "gemini-3-pro-image-preview"    (Nano Banana Pro: hi-res 1K/2K/4K)
MODEL_ID = "gemini-3-pro-image-preview"

image_agent = Agent(
    name="nano_banana_image_agent",
    model=MODEL_ID,
    instruction=(
        "You generate images from the user's prompt.\n"
        "Return an IMAGE only (no text)."
    ),
    generate_content_config=types.GenerateContentConfig(
        response_modalities=["IMAGE"],
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
            image_size="2K",  # Only applies to gemini-3-pro-image-preview
        ),
    ),
)

runner = InMemoryRunner(agent=image_agent)

async def generate_images(prompt: str, n: int = 1):
    saved_files = []

    for i in range(n):
        # Small trick to encourage variation across multiple calls:
        per_call_prompt = f"{prompt}\nVariation: {i+1} (make composition meaningfully different)."

        events = await runner.run_debug(per_call_prompt)

        # Find images in the final response event
        for ev in events:
            if ev.is_final_response() and ev.content and ev.content.parts:
                for part in ev.content.parts:
                    img = None
                    try:
                        img = part.as_image()
                    except Exception:
                        pass

                    if img is not None:
                        ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
                        fname = f"nano_banana_{ts}_{i+1}.png"
                        img.save(fname)
                        saved_files.append(fname)
                        display(img)

    return saved_files

files = await generate_images(
    prompt="A cinematic product photo of a minimalist matte-black water bottle on a stone pedestal, studio lighting, soft shadows.",
    n=1,  # set to 1 for a single image
)
files



 ### Created new session: debug_session_id

User > A cinematic product photo of a minimalist matte-black water bottle on a stone pedestal, studio lighting, soft shadows.
Variation: 1 (make composition meaningfully different).


Image(
  image_bytes=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x01,\x01,\x00\x00\xff\xeb\x17\x89JP\x00\x01\x00\x00\x00\x01\x00\x00\x17\x7fjumb\x00\x00\x00\x1ejumdc2pa\x00\x11\x00\x10\x80\x00\x00\xaa\x008\x9bq\x03c2pa\x00\x00\x00\x17Yjumb\x00\x00\x00Gjumdc2ma\x00\x11\x00\x10\x80\x00\x00...',
  mime_type='image/jpeg'
)


 ### Continue session: debug_session_id

User > A cinematic product photo of a minimalist matte-black water bottle on a stone pedestal, studio lighting, soft shadows.
Variation: 2 (make composition meaningfully different).


Image(
  image_bytes=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x01,\x01,\x00\x00\xff\xebU\x84JP\x00\x01\x00\x00\x00\x01\x00\x00Uzjumb\x00\x00\x00\x1ejumdc2pa\x00\x11\x00\x10\x80\x00\x00\xaa\x008\x9bq\x03c2pa\x00\x00\x00\x17Yjumb\x00\x00\x00Gjumdc2ma\x00\x11\x00\x10\x80\x00\x00...',
  mime_type='image/jpeg'
)

['nano_banana_20260224_155637_1.png', 'nano_banana_20260224_155706_2.png']

# Nano Banana with reference Image

In [7]:
from dotenv import load_dotenv
load_dotenv()

from google import genai
from google.genai import types
from PIL import Image
from IPython.display import display

# For mixing many reference images, the docs recommend using:
MODEL_ID = "gemini-3-pro-image-preview"

# --- Provide your reference images here ---
reference_paths = [
    "ref_1.png",
    "ref_2.png",
    # Add more paths as needed (up to 14 total for this model).
]

# Enforce the documented limit
if len(reference_paths) > 14:
    raise ValueError("Too many reference images. gemini-3-pro-image-preview supports up to 14 total.")

references = [Image.open(p) for p in reference_paths]

prompt = (
    "Create a premium e-commerce hero image.\n"
    "Use the reference product and keep its shape/material faithful.\n"
    "Place it on a clean white seamless background with soft studio lighting.\n"
    "Add subtle shadowing and realistic reflections.\n"
)

client = genai.Client()

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[prompt, *references],
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
            image_size="2K",  # "1K", "2K", "4K"
        ),
    ),
)

# Print any text + display/save the image(s)
for part in response.parts:
    if part.text:
        print(part.text.strip())
    else:
        img = part.as_image()
        if img:
            display(img)
            img.save("nano_banana_reference_mix.png")

"Saved: nano_banana_reference_mix.png"


Image(
  image_bytes=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x01,\x01,\x00\x00\xff\xeb\x19\xf9JP\x00\x01\x00\x00\x00\x01\x00\x00\x19\xefjumb\x00\x00\x00\x1ejumdc2pa\x00\x11\x00\x10\x80\x00\x00\xaa\x008\x9bq\x03c2pa\x00\x00\x00\x19\xc9jumb\x00\x00\x00Gjumdc2ma\x00\x11\x00\x10\x80\x00\x00...',
  mime_type='image/jpeg'
)

'Saved: nano_banana_reference_mix.png'

# Generating Video with Veo 3.1 using reference image + text

In [12]:
from dotenv import load_dotenv
load_dotenv()

import time
from google import genai
from google.genai import types
from IPython.display import Video, display

client = genai.Client()

prompt = "A cinematic video of the subject with visible eyes and mouth, talking directly to the viewer as if speaking to us. Shot from multiple angles in the same video: start with a slow dolly-in from the front so we see the face clearly, then smoothly orbit or cut to a side angle, then a three-quarter view. The subject should be speaking—lips moving, expressive eyes, addressing the camera. Keep subtle wind movement, natural lighting, and realistic audio ambience (including clear speech/voice). Show the subject from at least 3 distinct angles in one continuous or smoothly edited sequence."
# Veo API requires image with bytesBase64Encoded + mimeType; use types.Image.from_file (not PIL).
start_image = types.Image.from_file(location="ref_2.png")

operation = client.models.generate_videos(
    model="veo-3.1-generate-preview",
    prompt=prompt,
    image=start_image,
    config=types.GenerateVideosConfig(
        aspect_ratio="16:9",
        resolution="720p",
        negative_prompt="cartoon, low quality, blurry",
        # seed=42,  # The docs note seed exists for Veo 3 models; if your SDK errors on this field, remove it.
    ),
)

# Poll until done
while not operation.done:
    print("Waiting for Veo generation...")
    time.sleep(10)
    operation = client.operations.get(operation)

# Download the generated video (must download within 2 days per docs)
generated = operation.response.generated_videos[0]
client.files.download(file=generated.video)
generated.video.save("veo3_1_image_to_video.mp4")

display(Video("veo3_1_image_to_video.mp4", embed=True))
"Saved: veo3_1_image_to_video.mp4"


Waiting for Veo generation...
Waiting for Veo generation...
Waiting for Veo generation...
Waiting for Veo generation...
Waiting for Veo generation...
Waiting for Veo generation...
Waiting for Veo generation...
Waiting for Veo generation...


'Saved: veo3_1_image_to_video.mp4'